In [1]:
!pip install -q nltk scikit-learn ipywidgets


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\ARNAV\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [4]:
import re
import numpy as np
%pip install -q scikit-learn nltk ipywidgets
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import ipywidgets as widgets
from IPython.display import display

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 145] The directory is not empty: 'c:\\users\\arnav\\miniforge3\\lib\\site-packages\\numpy\\testing\\tests'



ModuleNotFoundError: No module named 'sklearn'

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\Users\\ARNAV\\miniforge3\\Lib\\site-packages\\numpy-2.5.4.dist-info\\INSTALLERe8crh9hm.tmp'



In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

In [ ]:
taxonomy = {
    "Diagnosis and Symptoms": ["Clinical Symptoms", "Diagnostic Biomarkers"],
    "Medical Management": ["Pharmacotherapy", "Glucose Monitoring"],
    "Lifestyle and Nutrition": ["Medical Nutrition Therapy", "Physical Activity"],
    "Complications and Emergencies": ["Acute Glycemic Crises", "Chronic Complications"]
}

# step 3: ontology using nested dictionary structure from lecture slides
ontology = {
    "Pancreas": {"secretes": "Insulin"},
    "Insulin": {"regulates": "BloodGlucose"},
    "Carbohydrates": {"increase": "BloodGlucose"},
    "Metformin": {"reduces": "BloodGlucose"},
    "Patient": {
        "monitors": "BloodGlucose",
        "administers": "Insulin",
        "takes": "Metformin"
    },
    "Low BloodGlucose": {"triggers": "Hypoglycemia"},
    "Severe Insulin Deficiency": {"causes": "DiabeticKetoacidosis"},
    "Hypoglycemia": {"requires": "FastActingSugar"}
}

# step 1: labelled corpus across 5 intent categories
dataset = [
    # symptoms intent
    ("What are some early warning signs of diabetes to watch out for?", "symptoms"),
    ("Why am I feeling excessively thirsty and urinating so often lately?", "symptoms"),
    ("Could blurry vision and chronic fatigue be symptoms of high blood sugar?", "symptoms"),
    ("Can unexplained weight loss indicate diabetes onset?", "symptoms"),
    ("I have tingling sensations in my feet and hands, could it be diabetes?", "symptoms"),
    ("What does hyperglycemia actually feel like physically?", "symptoms"),
    ("Why do cuts and sores heal very slowly on my skin?", "symptoms"),
    ("Can dry mouth and itchy skin be symptoms of high glucose?", "symptoms"),
    ("How can I tell if my blood sugar is dropping dangerously low?", "symptoms"),
    # diet and nutrition intent
    ("What foods should a type 2 diabetic strictly avoid eating?", "diet_nutrition"),
    ("Can I still eat fruits like apples and berries with diabetes?", "diet_nutrition"),
    ("How does carbohydrate counting actually work for meal planning?", "diet_nutrition"),
    ("Is intermittent fasting safe for someone managing diabetes?", "diet_nutrition"),
    ("What is the glycemic index, and how does it affect blood sugar?", "diet_nutrition"),
    ("Can diabetics consume artificial sweeteners safely on a regular basis?", "diet_nutrition"),
    ("What does a healthy balanced dinner plate for diabetes actually look like?", "diet_nutrition"),
    ("Do I need to cut all carbs completely to control blood glucose?", "diet_nutrition"),
    ("Are whole grains and oats actually good for blood sugar regulation?", "diet_nutrition"),
    # medication intent
    ("How does metformin help control type 2 diabetes in the body?", "medication"),
    ("What are the most common side effects of metformin when starting out?", "medication"),
    ("When is the best time to take long acting insulin injections during the day?", "medication"),
    ("Is it okay if I skip my insulin dose if I skip lunch?", "medication"),
    ("What should I do if I forget to take my diabetes tablet today?", "medication"),
    ("How does rapid acting insulin differ from basal insulin in practice?", "medication"),
    ("Are there non insulin injectable medications for diabetes available?", "medication"),
    ("Can diabetes medication cause stomach upset or nausea like this?", "medication"),
    ("How should unopened insulin pens and vials be stored properly?", "medication"),
    # blood glucose monitoring intent
    ("What is considered the ideal fasting blood sugar range for adults?", "monitoring"),
    ("How often should I check my glucose with a finger prick meter each day?", "monitoring"),
    ("What does an HbA1c test result of 7 percent actually mean?", "monitoring"),
    ("What should my post meal blood glucose reading be after 2 hours?", "monitoring"),
    ("How does a continuous glucose monitor track sugar levels throughout the day?", "monitoring"),
    ("Why does my blood sugar spike early in the morning before breakfast?", "monitoring"),
    ("What blood sugar number indicates severe hypoglycemia?", "monitoring"),
    ("How frequently should a diabetic get an HbA1c lab test done?", "monitoring"),
    ("What is the normal blood sugar target before going to bed at night?", "monitoring"),
    # prevention and lifestyle intent
    ("Can I really reverse prediabetes through lifestyle changes alone?", "prevention_lifestyle"),
    ("How much daily exercise is recommended for diabetes management in reality?", "prevention_lifestyle"),
    ("Does losing body weight improve insulin sensitivity significantly?", "prevention_lifestyle"),
    ("How exactly does chronic psychological stress raise blood glucose levels?", "prevention_lifestyle"),
    ("Can simple aerobic walking lower blood sugar levels immediately?", "prevention_lifestyle"),
    ("What daily lifestyle habits increase the risk of developing diabetes the most?", "prevention_lifestyle"),
    ("Does getting adequate sleep help regulate blood glucose?", "prevention_lifestyle"),
    ("How does strength training help my body process insulin better?", "prevention_lifestyle"),
    ("What should I do to prevent diabetic complications in feet and eyes?", "prevention_lifestyle")
]

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(lemmas)

# prepare corpus features and labels
queries = [preprocess_text(item[0]) for item in dataset]
labels = [item[1] for item in dataset]

# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(queries, labels, test_size=0.25, random_state=42, stratify=labels)

# step 5: feature extraction using tf idf and model training
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# train logistic regression classifier
model = LogisticRegression(random_state=42, max_iter=200)
model.fit(X_train_tfidf, y_train)

# evaluate model performance
y_pred = model.predict(X_test_tfidf)
print("Classification Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# step 6: domain knowledge base retrieval mapping
knowledge_base = {
    "symptoms": "Common diabetes symptoms include frequent urination (polyuria), excessive thirst (polydipsia), chronic fatigue, blurred vision, and slow-healing sores. Consult a healthcare provider for diagnostic laboratory tests.",
    "diet_nutrition": "Prioritize complex carbohydrates, high-fiber leafy vegetables, lean proteins, and low glycemic index foods. Minimize refined sugars and sweetened beverages to avoid post-meal glucose spikes.",
    "medication": "Diabetes medications (e.g., Metformin) and insulin facilitate glucose uptake. Always administer prescribed doses at consistent daily intervals and store insulin under refrigeration.",
    "monitoring": "Standard clinical targets for adults are 80-130 mg/dL fasting and under 180 mg/dL two hours post-meal. An HbA1c below 5.7% is normal, 5.7-6.4% indicates prediabetes, and 6.5%+ indicates diabetes.",
    "prevention_lifestyle": "Aim for at least 150 minutes of moderate aerobic exercise weekly combined with resistance training. A 5-7% sustained body weight reduction significantly improves insulin sensitivity."
}

In [ ]:
def assistant(query):
    query_lower = query.lower()

    # rule 1: emergency escalation
    emergency_triggers = ["unconscious", "passed out", "seizure", "collapsed", "cannot wake", "chest pain", "ketoacidosis", "dka", "difficulty breathing"]
    if any(trigger in query_lower for trigger in emergency_triggers):
        return "[EMERGENCY ALERT] Acute medical emergency detected. Call emergency medical services (911/112) immediately."

    # rule 2: critical glucose value escalation
    glucose_numbers = [int(num) for num in re.findall(r'\b\d{2,3}\b', query_lower)]
    for value in glucose_numbers:
        if value < 55:
            return f"[CRITICAL HYPOGLYCEMIA ALERT] Glucose reading of {value} mg/dL is dangerously low. Consume 15g fast-acting sugar immediately and recheck in 15 minutes."
        elif value > 350:
            return f"[CRITICAL HYPERGLYCEMIA ALERT] Glucose reading of {value} mg/dL is severely elevated. Contact your healthcare provider promptly."

    # rule 3: unsupported out of domain queries
    unsupported_topics = ["fracture", "broken bone", "plaster", "tooth", "dental", "cavity", "cancer", "appendix", "asthma inhaler", "sprain"]
    if any(topic in query_lower for topic in unsupported_topics):
        return "[UNSUPPORTED QUERY] This assistant specializes exclusively in Diabetes Care. Please consult an appropriate medical specialist."

    # rule 4: prescription modification safeguard
    dosage_alteration_words = ["stop my insulin", "change my dose", "double my dose", "adjust my dose", "quit metformin"]
    if any(phrase in query_lower for phrase in dosage_alteration_words):
        return "[PRESCRIPTION SAFEGUARD] Medication dosages must only be altered under direct medical supervision."

    # rule 5: intent classification with low confidence handling
    processed_query = preprocess_text(query)
    query_vector = vectorizer.transform([processed_query])
    probabilities = model.predict_proba(query_vector)[0]
    max_confidence = np.max(probabilities)
    predicted_intent = model.classes_[np.argmax(probabilities)]

    if max_confidence < 0.35:
        return f"[LOW CONFIDENCE ({max_confidence:.2f})] Could not reliably classify your query from the provided question. Please rephrase using specific diabetes terms."

    return f"[Intent: {predicted_intent} | Confidence: {max_confidence:.2f}]\n{knowledge_base[predicted_intent]}"

In [ ]:
# step 8: deployment with interactive colab widgets
input_box = widgets.Text(placeholder='Type your diabetes query here...', description='Query:', layout=widgets.Layout(width='70%'))
submit_btn = widgets.Button(button_style='primary', description='Ask Assistant')
output_area = widgets.Output()

def on_button_clicked(b):
    output_area.clear_output()
    with output_area:
        if input_box.value.strip():
            print(assistant(input_box.value))

submit_btn.on_click(on_button_clicked)
display(widgets.HBox([input_box, submit_btn]), output_area)